# Parameter Efficient Fine-Tuning with LoRa





This exercise uses code from the notebook authored by Achilles Moraites. The orignal notebook can be found  **[here](https://github.com/achimoraites/machine-learning-playground/blob/main/NLP/Text%20classification/Lightweight_RoBERTa_PEFT_LORA_FineTuning.ipynb)**. The Fine-tuning dataset is ag_news. We hope this notebook, with our added comments, serves as reference for you to understand the possiblities with LoRA.

######Copyright (c) 2021 Achilles Moraites



Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


## Install dependencies




In [ ]:
! pip install transformers datasets peft evaluate accelerate peft
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## Exercise #1: Dataset Preprocessing

---




In [2]:
import torch
from transformers import RobertaModel, RobertaTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
from datasets import load_dataset


# Define model names
base_model = "roberta-base"
modified_base = "roberta-base-modified"
peft_model_name = "roberta-base-peft"

# Load dataset
dataset = load_dataset("wangrongsheng/ag_news")
print("dataset")
print(dataset)
print()

# Initialize tokenizer using the base RoBERTa model
tokenizer = RobertaTokenizer.from_pretrained(base_model)
print("tokenizer")
print(tokenizer)
print(type(tokenizer))
print()

# Define a preprocessing function
def preprocess_function(chunk):
    return tokenizer(chunk["text"], truncation=True)


# Apply preprocessing to the entire dataset in batches, removing the 'text' column
tokenized_dataset = dataset.map(preprocess_function, remove_columns=["text"], batched=True)


# Split the tokenized dataset into train, evaluation, and test datasets
train_dataset = tokenized_dataset['train']  # Training dataset
eval_dataset = tokenized_dataset['test'].shard(num_shards=2, index=0)  # Evaluation dataset (1st shard)
test_dataset = tokenized_dataset['test'].shard(num_shards=2, index=1)  # Test dataset (2nd shard)


# Extract the number of classess and their names
num_labels = dataset['train'].features['label'].num_classes
class_names = dataset["train"].features["label"].names
print(f"number of labels: {num_labels}")
print(f"the labels: {class_names}")

# Create an id2label mapping
# We will need this for our classifier.
id2label = {i: label for i, label in enumerate(class_names)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

... (6 outputs left out)

tokenizer
RobertaTokenizer(name_or_path='roberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
})
<class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>



Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

number of labels: 4
the labels: ['World', 'Sports', 'Business', 'Sci/Tech']


In [3]:
print(dataset['train'][0])
print(train_dataset[0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'label': 2, 'input_ids': [0, 28216, 312, 4, 6033, 44121, 3727, 20693, 5, 1378, 36, 1251, 43, 1201, 111, 7787, 12, 5727, 268, 6, 2298, 852, 18, 25564, 37457, 9484, 9, 9620, 12, 4469, 282, 2857, 6, 32, 1782, 2272, 456, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Training

Let's Train two models, one using LORA and the other with full fine-tuning.

Note the LORA training times and the number of trained parameters!

In [4]:
# use the same Training args for all models

# Define training arguments for model training
training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model and training results
    eval_strategy='steps',  # Evaluate on the validation set every 'steps' training steps
    learning_rate=5e-5,  # Initial learning rate for the optimizer
    num_train_epochs=1,  # Number of times to iterate over the entire training dataset
    per_device_train_batch_size=16,  # Batch size for each GPU during training
)


In [5]:
# Define a function to create a Trainer object for model training
def get_trainer(model):
    # Create a Trainer object with the specified arguments
    return Trainer(
        model=model,  # The model to be trained
        args=training_args,  # Training arguments for configuring the training process
        train_dataset=train_dataset,  # Training dataset
        eval_dataset=eval_dataset,  # Evaluation dataset
        data_collator=data_collator,  # Data collator for batch processing
    )

In [ ]:

# Create a Trainer object for full fine-tuning of the model
full_finetuning_trainer = get_trainer(
    AutoModelForSequenceClassification.from_pretrained(base_model, id2label=id2label),
)
# Start training the model using the Trainer object
full_finetuning_trainer.train()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss
500,0.412615,0.395408
1000,0.322881,0.298753
1500,0.300708,0.376916
2000,0.275307,0.325194
2500,0.269490,0.281778
3000,0.266232,0.226847
3500,0.234753,0.232673
4000,0.232792,0.238847
4500,0.217983,0.219384
5000,0.211544,0.199371


... (12 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7500, training_loss=0.24972552286783853, metrics={'train_runtime': 3462.6001, 'train_samples_per_second': 34.656, 'train_steps_per_second': 2.166, 'total_flos': 5974931802837504.0, 'train_loss': 0.24972552286783853, 'epoch': 1.0})

In [ ]:
!zip -r my_trained_roberta.zip ./my_trained_roberta


  adding: my_trained_roberta/ (stored 0%)
  adding: my_trained_roberta/tokenizer_config.json (deflated 51%)
  adding: my_trained_roberta/tokenizer.json (deflated 82%)
  adding: my_trained_roberta/model.safetensors (deflated 8%)
  adding: my_trained_roberta/config.json (deflated 51%)
  adding: my_trained_roberta/training_args.bin (deflated 54%)


In [ ]:
save_path = "./my_trained_roberta"
full_finetuning_trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Model aur Tokenizer safely save ho gaye!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model aur Tokenizer safely save ho gaye!


In [ ]:
model_name = "mananjainz/roberta-agnews-classifier"
saved_model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
test_text = "AI godfather says Dario Amodei has medieval mindset, left OpenAI when things went out of control"

testing_tokenizer_obj = tokenizer(test_text, return_tensors = "pt", truncation = True, padding = True)

saved_model.eval()

with torch.no_grad():
  output = saved_model(**testing_tokenizer_obj)

prediction = output.logits.argmax(dim=-1).item()

print("Scores:", output.logits)
print("Category:", id2label[prediction])


Scores: tensor([[-0.8903, -4.0333,  0.0528,  4.5227]])
Category: Sci/Tech


## Exercise #2: PEFT Training

In [6]:

# Creating a model for sequence classification based on a pretrained model
model = AutoModelForSequenceClassification.from_pretrained(base_model, id2label=id2label)

# Configuring Lora parameters for PEFT (Proxy Experience Finetuning)
peft_config = LoraConfig( task_type = "SEQ_CLS", r=8, lora_alpha=16, lora_dropout=0.1)

# Applying Lora parameters to the model to create the PEFT model
peft_model = get_peft_model(model, peft_config)

# Printing information about the PEFT model's trainable parameters
peft_model.print_trainable_parameters()

# Obtaining a trainer object for Lora finetuning
peft_trainer = get_trainer(peft_model)

# Initiating the training process for the PEFT model using Lora finetuning
peft_trainer.train()


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 888,580 || all params: 125,537,288 || trainable%: 0.7078


Step,Training Loss,Validation Loss
500,0.616502,0.325758
1000,0.279987,0.327674
1500,0.278674,0.302197
2000,0.263863,0.300320
2500,0.269691,0.291077
3000,0.278782,0.272921
3500,0.253013,0.273040
4000,0.255528,0.275629
4500,0.253548,0.270359
5000,0.243528,0.266044


TrainOutput(global_step=7500, training_loss=0.2789903686523437, metrics={'train_runtime': 1953.9344, 'train_samples_per_second': 61.415, 'train_steps_per_second': 3.838, 'total_flos': 6036919257375744.0, 'train_loss': 0.2789903686523437, 'epoch': 1.0})

In [7]:
# Save
tokenizer.save_pretrained(modified_base)
peft_model.save_pretrained(peft_model_name)

In [8]:
!zip -r lora_model.zip ./roberta-base-peft ./roberta-base-modified

  adding: roberta-base-peft/ (stored 0%)
  adding: roberta-base-peft/adapter_config.json (deflated 60%)
  adding: roberta-base-peft/adapter_model.safetensors (deflated 7%)
  adding: roberta-base-peft/README.md (deflated 66%)
  adding: roberta-base-modified/ (stored 0%)
  adding: roberta-base-modified/tokenizer_config.json (deflated 51%)
  adding: roberta-base-modified/tokenizer.json (deflated 82%)


## Performing Inference with a PEFT Model

It's time to have some fun putting our model to work!

In [11]:
from peft import AutoPeftModelForSequenceClassification
from transformers import AutoTokenizer


# LOAD the Saved PEFT model

inference_model = AutoPeftModelForSequenceClassification.from_pretrained(peft_model_name, id2label=id2label)
tokenizer = AutoTokenizer.from_pretrained(modified_base)

def classify(text):
    # Tokenize the input text, truncate if needed, pad sequences, and convert to PyTorch tensors
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    # Pass the tokenized input through the inference model to get predictions
    output = inference_model(**inputs)

    # Extract the predicted class index from the model output
    prediction = output.logits.argmax(dim=-1).item()

    print(f'\n Class: {prediction}, Label: {id2label[prediction]}, Text: {text}')
    # return id2label[prediction]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
classify( "Kederis proclaims innocence Olympic champion Kostas Kederis today left hospital ahead of his date with IOC inquisitors claiming his ...")
classify( "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.")


 Class: 1, Label: Sports, Text: Kederis proclaims innocence Olympic champion Kostas Kederis today left hospital ahead of his date with IOC inquisitors claiming his ...

 Class: 2, Label: Business, Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindlingand of ultra-cynics, are seeing green again.


### Evaluate Models

To measure the improvement of the Training process we will need a baseline; let's compare the trained models with an untrained one.

Take a Look how the trained models perform

1.   The trained models against the untrained one
2.   The PEFT Model vs the regular fine-tuned one  



In [13]:

# Import necessary modules
from torch.utils.data import DataLoader  # DataLoader for creating data iterators
import evaluate  # Import module for computing evaluation metrics
from tqdm import tqdm  # tqdm for progress tracking during evaluation

# Load the evaluation metric (e.g., accuracy) from the evaluate module
metric = evaluate.load('accuracy')

# Define a function to evaluate the model on a given dataset
def evaluate_model(inference_model, dataset):
    # Create a DataLoader for the evaluation dataset with batch size 8 and using data_collator for batch processing
    eval_dataloader = DataLoader(dataset.rename_column("label", "labels"), batch_size=8, collate_fn=data_collator)
    # Determine the device (GPU or CPU) for model inference
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Move the inference model to the appropriate device (GPU or CPU)
    inference_model.to(device)
    # Set the model in evaluation mode (no gradient computation)
    inference_model.eval()

    # Iterate over batches in the evaluation dataloader
    for step, batch in enumerate(tqdm(eval_dataloader)):
        # Move the batch to the device (GPU or CPU)
        batch.to(device)
        # Disable gradient calculation for inference
        with torch.no_grad():
            # Perform inference using the inference model on the batch
            outputs = inference_model(**batch)
        # Extract the predicted class indices from the model outputs
        predictions = outputs.logits.argmax(dim=-1)
        # Assign predictions and reference labels to variables for metric computation
        predictions, references = predictions, batch["labels"]
        # Add the batch of predictions and references to the evaluation metric
        metric.add_batch(
            predictions=predictions,
            references=references,
        )

    # Compute the evaluation metric (e.g., accuracy) based on added batches
    eval_metric = metric.compute()
    # Print the computed evaluation metric
    print(eval_metric)



In [14]:
# Evaluate the non fine-tuned model
evaluate_model(AutoModelForSequenceClassification.from_pretrained(base_model, id2label=id2label), test_dataset)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 475/475 [00:19<00:00, 24.26it/s]

{'accuracy': 0.24236842105263157}


In [15]:
# Evaluate the PEFT fine-tuned model
evaluate_model(inference_model, test_dataset)

100%|██████████| 475/475 [00:20<00:00, 22.97it/s]

{'accuracy': 0.9297368421052632}


In [19]:
# Evaluate the Fully fine-tuned model
# evaluate_model(full_finetuning_trainer.model, test_dataset)
full_model = AutoModelForSequenceClassification.from_pretrained("mananjainz/roberta-agnews-classifier")
evaluate_model(full_model, test_dataset)

config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 475/475 [00:20<00:00, 23.68it/s]


{'accuracy': 0.9473684210526315}
